# Your first kernel: vector add on a T4

Companion to the card **first-kernel.html** (perf-3-kernels). It rebuilds Mark Harris's
"An Even Easier Introduction to CUDA (Updated)" ladder on your own Colab GPU.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

What you will measure (y[i] = x[i] + y[i] on N = 2^20 floats, 12,582,912 bytes moved per run):

1. one thread, `<<<1, 1>>>`
2. one block of 256 threads, `<<<1, 256>>>`
3. many blocks with a grid-stride loop, `<<<4096, 256>>>`
4. the same many-blocks kernel **without** prefetching unified memory (page faults)
5. the same kernel with explicit memory (`cudaMalloc` + `cudaMemcpy`), plus the copy times
6. a broken launch (2,048 threads per block) that `CUDA_CHECK` catches, and an asynchronous error that only shows up at `cudaDeviceSynchronize`

Every kernel is timed with `cudaEvent` after a warm-up run (median of 5), and every result is checked (max error must be 0).
Bandwidth = 3 arrays × N × 4 bytes ÷ time (read x, read y, write y), compared with the T4's 320 GB/s peak.
Colab's T4 clocks and availability vary, so expect your numbers to move a little from run to run.

In [ ]:
!nvidia-smi

In [ ]:
%%writefile common.h
#pragma once
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <vector>
#include <algorithm>
#include <cuda_runtime.h>

// The CUDA guide's CUDA_CHECK macro (section 2.1.7), plus exit() so a
// failure stops the program instead of printing and carrying on.
#define CUDA_CHECK(expr_to_check) do {                           \
    cudaError_t result = expr_to_check;                          \
    if (result != cudaSuccess) {                                 \
        fprintf(stderr, "CUDA Runtime Error: %s:%i:%d = %s\n",   \
                __FILE__, __LINE__, (int)result,                 \
                cudaGetErrorString(result));                     \
        exit(EXIT_FAILURE);                                      \
    }                                                            \
} while (0)

// Print-only version, used to show the error state without exiting.
#define CUDA_REPORT(expr) do {                                   \
    cudaError_t r_ = (expr);                                     \
    printf("  %-28s -> %s (%d)\n", #expr,                        \
           cudaGetErrorString(r_), (int)r_);                     \
} while (0)

const double T4_PEAK_GBPS = 320.0;

// cudaMemPrefetchAsync changed signature in CUDA 13.
inline void prefetch_to_gpu(const void* p, size_t bytes, int dev) {
#if CUDART_VERSION >= 13000
    cudaMemLocation loc = {};
    loc.type = cudaMemLocationTypeDevice;
    loc.id = dev;
    CUDA_CHECK(cudaMemPrefetchAsync(p, bytes, loc, 0, 0));
#else
    CUDA_CHECK(cudaMemPrefetchAsync(p, bytes, dev, 0));
#endif
}

// The three kernels from the post.
__global__ void add_one_thread(int n, float* x, float* y) {
    for (int i = 0; i < n; i++)
        y[i] = x[i] + y[i];
}

__global__ void add_one_block(int n, float* x, float* y) {
    int index = threadIdx.x;
    int stride = blockDim.x;
    for (int i = index; i < n; i += stride)
        y[i] = x[i] + y[i];
}

__global__ void add_grid(int n, float* x, float* y) {
    int index = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    for (int i = index; i < n; i += stride)
        y[i] = x[i] + y[i];
}

// Resets y on the GPU between timed runs (not timed).
__global__ void fill(int n, float* a, float v) {
    int index = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    for (int i = index; i < n; i += stride)
        a[i] = v;
}

inline void launch_add(int which, int blocks, int threads,
                       int n, float* x, float* y) {
    if (which == 0)      add_one_thread<<<blocks, threads>>>(n, x, y);
    else if (which == 1) add_one_block<<<blocks, threads>>>(n, x, y);
    else                 add_grid<<<blocks, threads>>>(n, x, y);
}

inline float median(std::vector<float> v) {
    std::sort(v.begin(), v.end());
    return v[v.size() / 2];
}

// Warm-up once, then time `reps` runs with cudaEvent; y is reset to 2 on
// the GPU before every run so the answer is always 3.
inline float time_add(int which, int blocks, int threads,
                      int n, float* x, float* y, int reps) {
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    std::vector<float> ms;
    for (int r = -1; r < reps; r++) {          // r = -1 is the warm-up
        fill<<<(n + 255) / 256, 256>>>(n, y, 2.0f);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaEventRecord(start));
        launch_add(which, blocks, threads, n, x, y);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));
        float t = 0;
        CUDA_CHECK(cudaEventElapsedTime(&t, start, stop));
        if (r >= 0) ms.push_back(t);
    }
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    return median(ms);
}

inline float max_error(int n, const float* y) {
    float e = 0.0f;
    for (int i = 0; i < n; i++) e = fmaxf(e, fabsf(y[i] - 3.0f));
    return e;
}

inline void report(const char* name, double ns, size_t bytes,
                   double base_ns, float err) {
    double gbps = bytes / ns;                  // bytes per ns = GB/s
    printf("%-30s %13.0f ns %8.1fx %9.2f GB/s %6.1f%% of 320  max err %g\n",
           name, ns, base_ns / ns, gbps, 100.0 * gbps / T4_PEAK_GBPS, err);
}

inline void print_device() {
    cudaDeviceProp p;
    CUDA_CHECK(cudaGetDeviceProperties(&p, 0));
    printf("GPU: %s, %d SMs, CUDA runtime %d\n\n",
           p.name, p.multiProcessorCount, CUDART_VERSION);
}

## 1–4. The ladder with unified memory

`cudaMallocManaged` memory can be read by CPU and GPU; pages move on demand.
The first three rows prefetch x and y to the GPU (as in the post's summary table).
Row 4 initialises the arrays on the CPU right before the launch, so the kernel page-faults them across (the post measured this for the many-blocks kernel).

In [ ]:
%%writefile ladder.cu
#include "common.h"

int main() {
    print_device();
    const int N = 1 << 20;
    const size_t bytes_moved = 3ull * N * sizeof(float);
    const int reps = 5;
    float *x, *y;
    CUDA_CHECK(cudaMallocManaged(&x, N * sizeof(float)));
    CUDA_CHECK(cudaMallocManaged(&y, N * sizeof(float)));
    for (int i = 0; i < N; i++) { x[i] = 1.0f; y[i] = 2.0f; }

    int blockSize = 256;
    int numBlocks = (N + blockSize - 1) / blockSize;   // ceil-div: 4096
    printf("N = %d, bytes moved per run = %zu, numBlocks = %d\n\n",
           N, bytes_moved, numBlocks);

    const char* names[3] = {"1 thread <<<1,1>>>",
                            "1 block <<<1,256>>>",
                            "grid-stride <<<4096,256>>>"};
    int blocks[3]  = {1, 1, numBlocks};
    int threads[3] = {1, 256, blockSize};
    double base_ns = 0;
    for (int k = 0; k < 3; k++) {
        prefetch_to_gpu(x, N * sizeof(float), 0);
        prefetch_to_gpu(y, N * sizeof(float), 0);
        CUDA_CHECK(cudaDeviceSynchronize());
        float ms = time_add(k, blocks[k], threads[k], N, x, y, reps);
        CUDA_CHECK(cudaDeviceSynchronize());
        float err = max_error(N, y);   // host read: y migrates back
        double ns = ms * 1e6;
        if (k == 0) base_ns = ns;
        report(names[k], ns, bytes_moved, base_ns, err);
    }

    // 4. No prefetch: pages start on the CPU, the kernel faults them in.
    std::vector<float> ms;
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    for (int r = 0; r < reps; r++) {
        for (int i = 0; i < N; i++) { x[i] = 1.0f; y[i] = 2.0f; }
        CUDA_CHECK(cudaEventRecord(start));
        add_grid<<<numBlocks, blockSize>>>(N, x, y);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));
        float t = 0;
        CUDA_CHECK(cudaEventElapsedTime(&t, start, stop));
        ms.push_back(t);
    }
    float err = max_error(N, y);
    report("grid-stride, NO prefetch", median(ms) * 1e6, bytes_moved,
           base_ns, err);

    printf("\nHarris, T4, nsys: 91,811,206 / 2,049,034 / 47,520 ns"
           " (prefetch); 4,514,384 ns without prefetch.\n");
    CUDA_CHECK(cudaFree(x));
    CUDA_CHECK(cudaFree(y));
    return 0;
}

In [ ]:
!nvcc -O3 -arch=sm_75 -o ladder ladder.cu && ./ladder

### Optional: profile with Nsight Systems

The post timed kernels with `nsys` (its `nsys_easy` wrapper). Colab may not ship `nsys`; if it doesn't, the `cudaEvent` numbers above are your measurement.
With `nsys`, look for the `[CUDA memcpy Unified H2D]` / `D2H` rows: the post saw 64 H2D and 24 D2H migrations for the unprefetched kernel.

In [ ]:
!if command -v nsys >/dev/null 2>&1; then nsys profile -t cuda --stats=true -o ladder_prof -f true ./ladder; else echo "nsys is not installed on this runtime; use the cudaEvent timings above."; fi

## 5. Explicit memory: `cudaMalloc` + `cudaMemcpy`

The CUDA guide's second style (section 2.1.3.2): pinned host buffers (`cudaMallocHost`), device buffers (`cudaMalloc`), and copies you write yourself.
No page faults are possible, but now the copies over PCIe are visible. Compare them with the kernel time.

In [ ]:
%%writefile explicit.cu
#include "common.h"

int main() {
    print_device();
    const int N = 1 << 20;
    const size_t size = N * sizeof(float);
    const size_t bytes_moved = 3ull * N * sizeof(float);
    float *hx, *hy, *dx, *dy;
    CUDA_CHECK(cudaMallocHost(&hx, size));   // pinned host memory
    CUDA_CHECK(cudaMallocHost(&hy, size));
    for (int i = 0; i < N; i++) { hx[i] = 1.0f; hy[i] = 2.0f; }
    CUDA_CHECK(cudaMalloc(&dx, size));
    CUDA_CHECK(cudaMalloc(&dy, size));

    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    float h2d_ms = 0, d2h_ms = 0;

    CUDA_CHECK(cudaEventRecord(start));
    CUDA_CHECK(cudaMemcpy(dx, hx, size, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dy, hy, size, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));
    CUDA_CHECK(cudaEventElapsedTime(&h2d_ms, start, stop));

    int blockSize = 256;
    int numBlocks = (N + blockSize - 1) / blockSize;
    float k_ms = time_add(2, numBlocks, blockSize, N, dx, dy, 5);

    CUDA_CHECK(cudaEventRecord(start));
    CUDA_CHECK(cudaMemcpy(hy, dy, size, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));
    CUDA_CHECK(cudaEventElapsedTime(&d2h_ms, start, stop));

    float err = max_error(N, hy);
    double k_ns = k_ms * 1e6;
    report("explicit, grid-stride kernel", k_ns, bytes_moved, k_ns, err);
    printf("copy x,y to GPU (8 MiB):  %10.0f ns  %6.2f GB/s\n",
           h2d_ms * 1e6, 2.0 * size / (h2d_ms * 1e6));
    printf("copy y back (4 MiB):      %10.0f ns  %6.2f GB/s\n",
           d2h_ms * 1e6, 1.0 * size / (d2h_ms * 1e6));
    printf("kernel share of copies+kernel: %.1f%%\n",
           100.0 * k_ms / (h2d_ms + k_ms + d2h_ms));

    CUDA_CHECK(cudaFree(dx));
    CUDA_CHECK(cudaFree(dy));
    CUDA_CHECK(cudaFreeHost(hx));
    CUDA_CHECK(cudaFreeHost(hy));
    return 0;
}

In [ ]:
!nvcc -O3 -arch=sm_75 -o explicit explicit.cu && ./explicit

## 6a. A broken launch that `CUDA_CHECK` catches

A block may hold at most 1,024 threads. `<<<numBlocks, 2048>>>` is rejected at launch, and a `<<< >>>` launch returns nothing,
so the only way to see it is to check the error state right after the launch.
Depending on your CUDA version the message is `invalid configuration argument` or `invalid argument`
(the CUDA guide's own example printed `invalid argument` for a 4,096-thread block).

`cudaPeekAtLastError` reads the error state without clearing it; `cudaGetLastError` reads it and resets it to success.

In [ ]:
%%writefile broken.cu
#include "common.h"

int main() {
    print_device();
    const int N = 1 << 20;
    float *x, *y;
    CUDA_CHECK(cudaMalloc(&x, N * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&y, N * sizeof(float)));

    printf("launch add_grid<<<512, 2048>>> (too many threads per block)\n");
    add_grid<<<512, 2048>>>(N, x, y);
    CUDA_REPORT(cudaPeekAtLastError());   // sees it, keeps it
    CUDA_REPORT(cudaGetLastError());      // sees it, clears it
    CUDA_REPORT(cudaGetLastError());      // cleared: success

    printf("launch add_grid<<<4096, 256>>> (valid)\n");
    add_grid<<<4096, 256>>>(N, x, y);
    CUDA_REPORT(cudaGetLastError());
    CUDA_REPORT(cudaDeviceSynchronize());

    printf("\nnow the same bad launch through CUDA_CHECK (exits):\n");
    add_grid<<<512, 2048>>>(N, x, y);
    CUDA_CHECK(cudaGetLastError());
    printf("not reached\n");
    return 0;
}

In [ ]:
!nvcc -O3 -arch=sm_75 -o broken broken.cu && ./broken

## 6b. An asynchronous error shows up later

This launch is valid, so the check right after it says success. The kernel then writes through a null pointer while the CPU has already moved on.
The error only appears at the next call that waits for or asks the GPU, here `cudaDeviceSynchronize`.
The CUDA guide (2.1.7.2): errors from asynchronous work "will only be reported when the error state is examined next".
Watch whether the error keeps coming back on later calls: an illegal memory access breaks the whole CUDA context, so it runs in its own program.

In [ ]:
%%writefile async_error.cu
#include "common.h"

__global__ void write_through(float* p) {
    p[threadIdx.x] = 1.0f;       // p is null at run time
}

int main() {
    print_device();
    float* p = nullptr;
    write_through<<<1, 32>>>(p);
    printf("right after the launch:\n");
    CUDA_REPORT(cudaGetLastError());
    printf("after waiting for the GPU:\n");
    CUDA_REPORT(cudaDeviceSynchronize());
    printf("later calls:\n");
    CUDA_REPORT(cudaGetLastError());
    float* q = nullptr;
    CUDA_REPORT(cudaMalloc(&q, 4));
    return 0;
}

In [ ]:
!nvcc -O3 -arch=sm_75 -o async_error async_error.cu && ./async_error

## Reference numbers from the source

Mark Harris, *An Even Easier Introduction to CUDA (Updated)*, NVIDIA blog, May 2025 — **NVIDIA T4**, timed with nsys, N = 2^20, with `cudaMemPrefetchAsync`:

| Version | Time | Speedup | Bandwidth |
|---|---|---|---|
| Single thread | 91,811,206 ns | 1× | 137 MB/s |
| Single block (256 threads) | 2,049,034 ns | 45× | 6 GB/s |
| Multiple blocks | 47,520 ns | 1932× | 265 GB/s |

Without prefetch, the multiple-blocks kernel took 4,514,384 ns, with 64 unified H2D and 24 D2H memcpy operations.
265 GB/s is "over 80% of the T4's peak bandwidth of 320GB/s". Bandwidth here = 12,582,912 bytes ÷ time (our recomputation matches all three rows).

## Try this

- Set `numBlocks` to `32 * multiProcessorCount` (1,280 on a T4) instead of 4,096. The grid-stride loop still covers all N elements; does the time change?
- Change `N` to `1 << 26` (768 MiB moved per run). The kernel gets closer to 320 GB/s because fixed costs are spread over more bytes (the 1-thread row then takes seconds per run; skip it or lower `reps`).
- Change the block size to 32, 128, 512 and 1024. Anything above 1,024 fails at launch, as in section 6a.